# 4 — DCM vs CCM Side-by-Side Simulation

> **Goal.** With both designs validated independently, now run them
> on the same axes: steady-state at multiple line voltages, line-step
> response, load-step response, full-range PF/THD sweep. Synthesize
> the case for picking one or the other.

**Prerequisites**

- Notebooks 01, 02, 03 of this project.

This notebook reuses the compensators designed in the previous two
notebooks (briefly re-derived here for self-containedness) and runs
focused experiments.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
from dataclasses import replace

from boost_pfc_model import (
    BoostPFCParams,
    dcm_voltage_loop_plant, ccm_current_loop_plant, ccm_voltage_loop_plant,
    simulate_closed_loop_dcm, simulate_closed_loop_ccm,
    power_factor, thd_current, line_band_filter,
)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

p_dcm = BoostPFCParams.dcm_design()
p_ccm = BoostPFCParams.ccm_design()
V_ramp = 5.0
T_s = p_dcm.T_sw  # same f_sw for both
print(f"DCM: L = {p_dcm.L*1e6:.1f} µH")
print(f"CCM: L = {p_ccm.L*1e6:.1f} µH")


## 1. Re-derive the compensators

Quick reproduction of notebooks 02 and 03 compensators.


In [ ]:
def design_pi(plant, f_c, V_ramp=5.0, zero_freq=None):
    '''PI for a first-order plant K/(1+s·tau). Cancels the plant pole.'''
    plant_pwm = signal.TransferFunction(np.array(plant.num)/V_ramp, np.array(plant.den))
    omega_c = 2*np.pi*f_c
    if zero_freq is None:
        tau_z = plant.den[0]/plant.den[1]  # plant pole reciprocal
    else:
        tau_z = 1.0/(2*np.pi*zero_freq)
    dc_gain = plant_pwm.num[0]/plant_pwm.den[1]
    K = omega_c / dc_gain
    Gc = signal.TransferFunction([K*tau_z, K], [1.0, 0.0])
    bd, ad, _ = signal.cont2discrete((Gc.num, Gc.den), dt=T_s, method='bilinear')
    return np.asarray(bd).flatten()/ad[0], np.asarray(ad)/ad[0]

def design_pi_integrator(plant, f_c, V_ramp=5.0, zero_below_xover=3.0):
    '''PI for an integrator plant V_o/(sL). Zero placed below crossover.'''
    plant_pwm = signal.TransferFunction(np.array(plant.num)/V_ramp, np.array(plant.den))
    omega_c = 2*np.pi*f_c
    tau_z = zero_below_xover / omega_c  # zero at f_c/zero_below_xover
    plant_mag_at_fc = plant_pwm.num[0] / (omega_c * plant_pwm.den[0])
    K = omega_c / (np.sqrt(1+(omega_c*tau_z)**2) * plant_mag_at_fc)
    Gc = signal.TransferFunction([K*tau_z, K], [1.0, 0.0])
    bd, ad, _ = signal.cont2discrete((Gc.num, Gc.den), dt=T_s, method='bilinear')
    return np.asarray(bd).flatten()/ad[0], np.asarray(ad)/ad[0]

# DCM: 1 PI for voltage loop
bv_dcm, av_dcm = design_pi(dcm_voltage_loop_plant(p_dcm), f_c=10.0, V_ramp=V_ramp)
# CCM: 2 PIs (current + voltage) + multiplier
bi_ccm, ai_ccm = design_pi_integrator(ccm_current_loop_plant(p_ccm),
                                       f_c=10e3, V_ramp=V_ramp, zero_below_xover=3.0)
bv_ccm, av_ccm = design_pi(ccm_voltage_loop_plant(p_ccm), f_c=10.0, V_ramp=V_ramp)
K_mult = 0.001
print("Compensators designed.")


## 2. Steady-state at $V_{ac}$ = 120 V — both modes


In [ ]:
sim_dcm = simulate_closed_loop_dcm(p_dcm, bv_dcm, av_dcm, V_ac=120.0,
                                    n_line_cycles=15, samples_per_period=60,
                                    v_ref=400.0, V_ramp=V_ramp)
sim_ccm = simulate_closed_loop_ccm(p_ccm, bi_ccm, ai_ccm, bv_ccm, av_ccm,
                                    V_ac=120.0, n_line_cycles=40, samples_per_period=40,
                                    v_ref=400.0, V_ramp=V_ramp, K_mult=K_mult)

fig, axs = plt.subplots(3, 2, figsize=(15, 9), sharex=True)
for col, (sim, title, p_) in enumerate([(sim_dcm, "DCM", p_dcm), (sim_ccm, "CCM", p_ccm)]):
    axs[0, col].plot(sim['t']*1000, sim['v_o'], "C2", linewidth=1.0)
    axs[0, col].axhline(400.0, color="k", linestyle=":", alpha=0.4)
    axs[0, col].set_ylabel("$v_o$ [V]")
    axs[0, col].set_title(f"{title} — V_ac = 120 V, P_o = {p_.P_o:.0f} W")
    fs = 1.0/(sim['t'][1] - sim['t'][0])
    i_filt = line_band_filter(sim['i_in'], fs, p_.f_line, 20)
    axs[1, col].plot(sim['t']*1000, sim['i_in'], "C3", linewidth=0.3, alpha=0.4)
    axs[1, col].plot(sim['t']*1000, i_filt, "C1", linewidth=1.5, label="filtered")
    axs[1, col].set_ylabel("$i_{in}$ [A]")
    axs[1, col].legend()
    axs[2, col].plot(sim['t']*1000, sim['duty'], "C4", linewidth=0.5)
    axs[2, col].set_ylabel("Duty"); axs[2, col].set_xlabel("Time [ms]")

plt.tight_layout(); plt.show()

# Metrics
for sim, name, p_ in [(sim_dcm, "DCM", p_dcm), (sim_ccm, "CCM", p_ccm)]:
    mask = sim['t'] >= (30.0 if "CCM" in name else 6.0)/p_.f_line
    fs = 1.0/(sim['t'][1] - sim['t'][0])
    pf = power_factor(sim['v_ac'][mask], sim['i_in'][mask], f_s=fs, f_line=p_.f_line)
    i_filt_m = line_band_filter(sim['i_in'][mask], fs, p_.f_line, 20)
    thd = thd_current(i_filt_m, fs, p_.f_line)
    v_o_mean = sim['v_o'][mask].mean()
    iL_max = sim['i_L'][mask].max()
    print(f"{name}: V_o = {v_o_mean:.2f}V, PF = {pf:.4f}, THD = {thd*100:.2f}%, "
          f"i_L peak = {iL_max:.2f}A")


## 3. PF and THD across the universal-mains range


In [ ]:
V_ac_sweep = [90, 110, 130, 160, 200, 230]
results = {"DCM": {"PF": [], "THD": [], "iL_pk": []},
           "CCM": {"PF": [], "THD": [], "iL_pk": []}}
for V_ac in V_ac_sweep:
    s_d = simulate_closed_loop_dcm(p_dcm, bv_dcm, av_dcm, V_ac=V_ac,
                                    n_line_cycles=10, samples_per_period=60,
                                    v_ref=400.0, V_ramp=V_ramp)
    s_c = simulate_closed_loop_ccm(p_ccm, bi_ccm, ai_ccm, bv_ccm, av_ccm,
                                    V_ac=V_ac, n_line_cycles=40, samples_per_period=40,
                                    v_ref=400.0, V_ramp=V_ramp, K_mult=K_mult)
    for sim, name, p_ in [(s_d, "DCM", p_dcm), (s_c, "CCM", p_ccm)]:
        mask = sim['t'] >= (30.0 if name == "CCM" else 5.0)/p_.f_line
        fs = 1.0/(sim['t'][1] - sim['t'][0])
        pf = power_factor(sim['v_ac'][mask], sim['i_in'][mask], f_s=fs, f_line=p_.f_line)
        i_filt_m = line_band_filter(sim['i_in'][mask], fs, p_.f_line, 20)
        thd = thd_current(i_filt_m, fs, p_.f_line)
        results[name]["PF"].append(pf)
        results[name]["THD"].append(thd*100)
        results[name]["iL_pk"].append(sim['i_L'][mask].max())
    print(f"V_ac={V_ac}V done")

fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))
for col, key, ylabel, ylim in [
    (0, "PF", "Power factor", (0.85, 1.01)),
    (1, "THD", "THD [%]", (0, 35)),
    (2, "iL_pk", "Peak $i_L$ [A]", None),
]:
    axs[col].plot(V_ac_sweep, results["DCM"][key], "o-", color="C3", label="DCM")
    axs[col].plot(V_ac_sweep, results["CCM"][key], "s-", color="C0", label="CCM")
    axs[col].set_xlabel("$V_{ac,rms}$ [V]"); axs[col].set_ylabel(ylabel)
    if ylim: axs[col].set_ylim(ylim)
    axs[col].legend()
axs[0].axhline(0.95, color="C2", linestyle=":", alpha=0.5, label="IEC target 0.95")
axs[1].axhline(20, color="C2", linestyle=":", alpha=0.5)
plt.suptitle("PF, THD, and peak inductor current across universal mains")
plt.tight_layout(); plt.show()


## 4. Line step

What happens when the line voltage steps from 90 → 180 V mid-run?
The slow voltage loop has to re-regulate the output. Expect 100s of
ms of transient — far slower than the DC-DC converters we've seen,
because the BW is constrained to be << $f_{line}$.

We approximate this with a piecewise input: $V_{ac}$ = 90 V for the
first 200 ms, then 180 V for the next 200 ms. We use the simulator
with two consecutive runs and stitch.


In [ ]:
# Run DCM with a manually-implemented line step via two segments
# We need a stitched simulator; simplest is to run for 20 line cycles
# at 90V, snapshot final state, restart at 180V. The shipped simulator
# doesn't expose state IO, so we use a quick custom inline run.

# For now, just run each line voltage independently and show the SS
# performance at each — full line-step transient is a future extension.
print("Line-step transient: see the future-work item in the project README.")
print("Here we just report steady-state performance at the two endpoints:")
for V_ac in [90.0, 180.0]:
    s_d = simulate_closed_loop_dcm(p_dcm, bv_dcm, av_dcm, V_ac=V_ac,
                                    n_line_cycles=15, samples_per_period=60,
                                    v_ref=400.0, V_ramp=V_ramp)
    mask = s_d['t'] >= 10.0/p_dcm.f_line
    fs = 1.0/(s_d['t'][1]-s_d['t'][0])
    pf = power_factor(s_d['v_ac'][mask], s_d['i_in'][mask], f_s=fs, f_line=p_dcm.f_line)
    i_filt = line_band_filter(s_d['i_in'][mask], fs, p_dcm.f_line, 20)
    thd = thd_current(i_filt, fs, p_dcm.f_line)
    print(f"  DCM @ V_ac={V_ac:.0f}V: V_o={s_d['v_o'][mask].mean():.1f}V, "
          f"PF={pf:.4f}, THD={thd*100:.2f}%")


## 5. Load step

Halve the load mid-run: $P_o$ goes from 100 W to 50 W. Both modes
should re-regulate; the transient duration depends on the voltage-
loop bandwidth (same for both modes by design).


In [ ]:
# Approximation: run at half load (P_o=50W → R_load=3200Ω) and compare
p_dcm_half = replace(p_dcm, P_o=50.0)
p_ccm_half = replace(p_ccm, P_o=50.0)

# Re-design controllers for half-load plant (lower bandwidth penalty)
bv_dcm_h, av_dcm_h = design_pi(dcm_voltage_loop_plant(p_dcm_half), f_c=10.0, V_ramp=V_ramp)
# (current-loop plant is independent of P_o for CCM, voltage-loop changes)
bv_ccm_h, av_ccm_h = design_pi(ccm_voltage_loop_plant(p_ccm_half), f_c=10.0, V_ramp=V_ramp)

for label, p_, sim_fn, args in [
    ("DCM full load", p_dcm,    simulate_closed_loop_dcm,
        (bv_dcm, av_dcm)),
    ("DCM half load", p_dcm_half, simulate_closed_loop_dcm,
        (bv_dcm_h, av_dcm_h)),
    ("CCM full load", p_ccm,    simulate_closed_loop_ccm,
        (bi_ccm, ai_ccm, bv_ccm, av_ccm)),
    ("CCM half load", p_ccm_half, simulate_closed_loop_ccm,
        (bi_ccm, ai_ccm, bv_ccm_h, av_ccm_h)),
]:
    if "DCM" in label:
        s = sim_fn(p_, *args, V_ac=120.0, n_line_cycles=15, samples_per_period=40,
                   v_ref=400.0, V_ramp=V_ramp)
        mask = s['t'] >= 8.0/p_.f_line
    else:
        s = sim_fn(p_, *args, V_ac=120.0, n_line_cycles=40, samples_per_period=40,
                   v_ref=400.0, V_ramp=V_ramp, K_mult=K_mult)
        mask = s['t'] >= 30.0/p_.f_line
    fs = 1.0/(s['t'][1]-s['t'][0])
    pf = power_factor(s['v_ac'][mask], s['i_in'][mask], f_s=fs, f_line=p_.f_line)
    i_filt = line_band_filter(s['i_in'][mask], fs, p_.f_line, 20)
    thd = thd_current(i_filt, fs, p_.f_line)
    print(f"  {label}: V_o={s['v_o'][mask].mean():.1f}V, PF={pf:.4f}, "
          f"THD={thd*100:.2f}%, i_L_pk={s['i_L'][mask].max():.2f}A")


## 6. Pick a mode — design guide

After all this, the design choice for a real product:

**Pick DCM if**:
- Power ≤ 150 W
- BOM cost is tight (one less PI + multiplier in firmware)
- High switching frequency (>200 kHz) is OK (DCM benefits from
  it — keeps the inductor small)
- Some THD margin available (passes IEC class D easily at low line,
  marginal at high line)

**Pick CCM if**:
- Power ≥ 200 W
- Tight THD spec across full line range
- EMI is critical (CCM has continuous current → softer EMI signature)
- Mid-range $f_{sw}$ (50-100 kHz) is preferred for efficiency

**Transition-mode (CrM) PFC**, used in chips like L6562, is a third
option not covered here: variable $T_{sw}$ keeps $i_L$ on the
DCM/CCM boundary. Combines the natural-PFC property of DCM with the
lower peak current of CCM. The math is more involved and the
controller is mostly analog hardware logic; we skip it in this
notebook.

## Summary

Boost PFC is the gateway converter into the world of AC-input power
electronics. DCM and CCM offer two different philosophies — physics
vs active control — for solving the same problem (shape the input
current to look like a resistor at the line).

**Cross-validation against Pulsim**: open
[`00_boost_pfc_pulsim_validation.ipynb`](00_boost_pfc_pulsim_validation.ipynb)
for an executed notebook that builds the full AC-input power stage
in Pulsim (``add_sine_voltage_source`` → ``add_bridge_rectifier`` →
boost stage) and overlays the rectified line voltage against the
analytical $|V_{pk} \\sin(\\omega_{line} t)|$ envelope.

Both modes share:
- First-order voltage-loop plant
- 2·$f_{line}$ output ripple
- Voltage-loop bandwidth constrained to $\le 2 f_{line}/10$

They differ in:
- Inductor sizing (factor of 10×)
- Controller complexity (1 loop vs 2 loops + multiplier)
- PF/THD at high line (DCM degrades, CCM holds)
- Peak inductor current (DCM is 2-3× higher)

**Library now covers**:

| Converter | Settling | Input | Loops | Notes |
|---|---|---|---|---|
| buck → half-bridge (6 converters) | µs–ms | DC | 1 | |
| **boost PFC DCM** | **100 ms** | **AC** | **1** | **first-order plant!** |
| **boost PFC CCM** | **100 ms** | **AC** | **2** | **multiplier-based** |

**Suggested exercises**

1. Implement the missing line-step transient (run for 200 ms at
   90 V, snapshot state, continue at 230 V). Compare DCM vs CCM
   settling time.
2. Replace the PI compensators with a Type-II (add a high-frequency
   pole at 1 kHz). Does it help with switching ripple injection
   into the loop?
3. Push the voltage-loop bandwidth to 30 Hz on both modes and watch
   PF degrade as the loop "corrects" the 2·f_line ripple.
4. Add input filter (L + C between line and bridge) and watch how
   the filter resonance interacts with the PFC stage. (Hint: a
   2.2 µF cap and 0.5 mH inductor put a resonance at ~5 kHz — well
   below the current-loop bandwidth.)
